In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=0.001,
    line_search_method="const",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.36629974842071533
epoch:  1, loss: 0.3594103157520294
epoch:  2, loss: 0.352562814950943
epoch:  3, loss: 0.3457346260547638
epoch:  4, loss: 0.3390960693359375
epoch:  5, loss: 0.3325183391571045
epoch:  6, loss: 0.3260749578475952
epoch:  7, loss: 0.3198167383670807
epoch:  8, loss: 0.3137114346027374
epoch:  9, loss: 0.307714581489563
epoch:  10, loss: 0.3018318712711334
epoch:  11, loss: 0.2961007058620453
epoch:  12

KeyboardInterrupt: 

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9371787309646606
Test metrics:  R2 = 0.8765665292739868


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr=1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.0695197805762291
epoch:  1, loss: 0.06246122345328331
epoch:  2, loss: 0.049489136785268784
epoch:  3, loss: 0.04076942801475525
epoch:  4, loss: 0.03494040295481682
epoch:  5, loss: 0.031292010098695755
epoch:  6, loss: 0.015398476272821426
epoch:  7, loss: 0.009278041310608387
epoch:  8, loss: 0.006746743340045214
epoch:  9, loss: 0.005301239900290966
epoch:  10, loss: 0.0041214823722839355
epoch:  11, loss: 0.0034826865885406733
epoch:  12, loss: 0.003091552061960101
epoch:  13, loss: 0.002829485572874546
epoch:  14, loss: 0.0026337841991335154
epoch:  15, loss: 0.002447430742904544
epoch:  16

KeyboardInterrupt: 

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9391120076179504
Test metrics:  R2 = 0.6806579232215881


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="interpolate",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.10417373478412628
epoch:  1, loss: 0.10417373478412628
epoch:  2, loss: 0.10417373478412628
epoch:  3, loss: 0.10417373478412628
epoch:  4, loss: 0.10417373478412628
epoch:  5, loss: 0.10417373478412628
epoch:  6, loss: 0.10417373478412628
epoch:  7, loss: 0.10417373478412628
epoch:  8, loss: 0.10417373478412628
epoch:  9, loss: 0.10417373478412628
epoch:  10, loss: 0.10417373478412628
epoch:  11, loss: 0.10417373478412628
epoch:  12, loss: 0.10417373478412628
epoch:  13, loss: 0.10417373478412628
epoch:  14, loss: 0.10417373478412628
epoch:  15, loss: 0.10417373478412628
epoch:  16

KeyboardInterrupt: 

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -1338.7403564453125
Test metrics:  R2 = -1349.729248046875


In [8]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="bisect",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3052538335323334
epoch:  1, loss: 0.2528351843357086
epoch:  2, loss: 0.24066179990768433
epoch:  3, loss: 0.11440610885620117
epoch:  4, loss: 0.06059980019927025
epoch:  5, loss: 0.038201846182346344
epoch:  6, loss: 0.03528245538473129
epoch:  7, loss: 0.03497997298836708
epoch:  8, loss: 0.022904464974999428
epoch:  9, loss: 0.025073010474443436
epoch:  10, loss: 0.021381206810474396
epoch:  11, loss: 0.017068728804588318
epoch:  12, loss: 0.013619928620755672
epoch:  13, loss: 0.007904864847660065
epoch:  14, loss: 0.006992620415985584
epoch:  15, loss: 0.006309109274297953
epoch:  16, loss: 0.005736848339438438
epoch:  17, loss: 0.005342632532119751
epoch:  18, loss: 0.005073571112006903
epoch:  19, loss: 0.004950781352818012
epoch:  20, loss: 0.12791834771633148
epoch:  21, loss: 10.389808654785156
epoch:  22, loss: 8.035102844238281
epoch:  23, loss: 0.027767641469836235
epoch:  24, loss: 0.013905743137001991
epoch:  25, loss: 0.006237514782696962
epoch:  26,

KeyboardInterrupt: 

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8652938604354858
Test metrics:  R2 = 0.5300534963607788
